# Charging / Refueling Infrastructure Subsidy → Service Subsidy (GCAM input)

This notebook converts:

1) **EV charging infrastructure subsidy** (per charger CAPEX subsidy, utilization → kWh throughput)
2) **Hydrogen refueling infrastructure subsidy** (per station CAPEX subsidy, H2 throughput → kg-H2)

into **service-level subsidies**:

- Passenger cars / buses: USD per **passenger-km**
- Medium trucks: USD per **ton-km**

Outputs are written as GCAM policy XML with:
- policy name: `charging-infra-subsidy`
- years: 2025, 2030, 2035
- constant USD in `BASE_INPUT` (model base year, e.g., 1990)


## How CP and EP are defined

We compute two policy variants:

### Current Policies (CP)
- Uses **multi-year average** of the policy unit subsidy in the recent period.
- For this exercise:
  - EV chargers: average over 2022–2024 (per-charger subsidy schedules)
  - H2 stations: average of **yearly weighted-average** subsidies over 2022–2024

### Enhanced Policies / High Ambition (EP)
- Uses the **peak** unit subsidy within an expanded window.
- For this exercise:
  - EV chargers: peak over 2022–2025 (since 2025 rule is explicitly available)
  - H2 stations: peak over 2022–2024 (we only digitized 2022–2024 tables)

All values are converted into **constant USD(BASE_INPUT)** using:
- KRW/USD exchange rate (`dfExcAnnual`)
- GDP deflator (`arrDef`), with: USD_const(base) = USD_nom × (DEF_base / DEF_year)


2010년 자료 기준 버스 평균 재차 인원 (15.03 - 시내버스, 11.40 - 시외/고속버스, 여기서는 시내 버스 외에 전세버스도 시외/고속버스로 가정)
2025년 자료 기준 버스 비율 (시내: 41153 / (41153 + 24368 + 38357 + 16))

In [16]:
41153 / (41153 + 24368 + 38357 + 16)

0.3961056461393343

In [17]:
import numpy as np
import pandas as pd

from utils import (
    pv_factor,
    tenkkrw_series_to_usd_base,
    baekmanwon_to_usd_base,
    yearly_weighted_average,
    items_from_perf_df,
    merge_items_sum,
    make_records,
    records_to_gcam_xml,
)

from build_h2_station_subsidy_df import build_h2_station_df_2022_2024

In [18]:
BASE_INPUT = 1990
r = 0.045

YEARS_XML = [2025, 2030, 2035]

years_cp = [2022, 2023, 2024]
years_ha = [2022, 2023, 2024, 2025]   # EV charger series includes 2025

In [19]:
dfDef = pd.read_csv("../resources/gdpdef.csv")
dfDef.head()
dfDef['Year'] = dfDef['date'].str.split('-').str[0].astype(int)
arrDef = dfDef.set_index('Year')['gdpdef']

dfExc = pd.read_csv("../resources/DEXKOUS.csv")
dfExc['date'] = pd.to_datetime(dfExc['observation_date'])
dfExcAnnual = dfExc.groupby(dfExc['date'].dt.year)['DEXKOUS'].mean()

In [20]:
# --------------------------
# 1) EV charger unit subsidies (만원 -> constant USD base)
# --------------------------
s_fast_10kkrw = pd.Series({2022: 2000, 2023: 2000, 2024: 2000, 2025: 2600}, dtype=float)
s_slow_10kkrw = pd.Series({2022: 16,   2023: 14,   2024: 18,   2025: 22},   dtype=float)

s_fast_usd = tenkkrw_series_to_usd_base(s_fast_10kkrw, base_year=BASE_INPUT, dfExcAnnual=dfExcAnnual, arrDef=arrDef)
s_slow_usd = tenkkrw_series_to_usd_base(s_slow_10kkrw, base_year=BASE_INPUT, dfExcAnnual=dfExcAnnual, arrDef=arrDef)

sub_fast_cp = float(s_fast_usd.loc[years_cp].mean())
sub_slow_cp = float(s_slow_usd.loc[years_cp].mean())

sub_fast_ep = float(s_fast_usd.loc[years_ha].max())
sub_slow_ep = float(s_slow_usd.loc[years_ha].max())

print(f"EV charger subsidy (constant USD base {BASE_INPUT})")
print(" Fast CP:", sub_fast_cp, "Slow CP:", sub_slow_cp)
print(" Fast EP:", sub_fast_ep, "Slow EP:", sub_slow_ep)

EV charger subsidy (constant USD base 1990)
 Fast CP: 7377.722071865814 Slow CP: 58.861582478860264
 Fast EP: 8212.99954360641 Slow EP: 69.49461152282346


In [21]:
# --------------------------
# 2) Utilization -> annual kWh per charger
# --------------------------
fast_kw, fast_n, fast_minutes_day_all = 50, 1974, 106_198
slow_kw, slow_n, slow_minutes_day_all = 7,  26_259, 2_001_958

alpha_fast, alpha_slow = 0.7, 0.85

fast_min_day_per = fast_minutes_day_all / fast_n
slow_min_day_per = slow_minutes_day_all / slow_n

fast_kwh_year_per = fast_kw * alpha_fast * (fast_min_day_per / 60) * 365
slow_kwh_year_per = slow_kw * alpha_slow * (slow_min_day_per / 60) * 365

In [22]:
# --------------------------
# 3) PV lifetime throughput (kWh) per charger
# --------------------------
L_fast, L_slow = 10, 15
fast_kwh_pv = fast_kwh_year_per * pv_factor(L_fast, r)
slow_kwh_pv = slow_kwh_year_per * pv_factor(L_slow, r)

In [23]:
# --------------------------
# 4) Blended USD per PV-kWh (network weighted)
# --------------------------
fast_total_pv_kwh = fast_kwh_pv * fast_n
slow_total_pv_kwh = slow_kwh_pv * slow_n

usdperkwh_blend_cp = (sub_fast_cp * fast_n + sub_slow_cp * slow_n) / (fast_total_pv_kwh + slow_total_pv_kwh)
usdperkwh_blend_ep = (sub_fast_ep * fast_n + sub_slow_ep * slow_n) / (fast_total_pv_kwh + slow_total_pv_kwh)

print("EV infra: blended USD/kWh (CP):", usdperkwh_blend_cp)
print("EV infra: blended USD/kWh (EP):", usdperkwh_blend_ep)

EV infra: blended USD/kWh (CP): 0.016830775080184383
EV infra: blended USD/kWh (EP): 0.018845179454594234


In [24]:
# --------------------------
# 5) USD/kWh -> USD per service (pass-km or ton-km)
# --------------------------
# efficiencies: kWh per vehicle-km (replace with your final values)
eps_car, eps_bus, eps_truck = 1/4.2, 1/1.08, 1/1.0

LF_car = 1.26
LF_bus = 15.03 * 0.3961 + 11.40 * (1-0.3961)
TL_truck = 4.2

ev_cp_car = usdperkwh_blend_cp * eps_car / LF_car
ev_cp_bus = usdperkwh_blend_cp * eps_bus / LF_bus
ev_cp_trk = usdperkwh_blend_cp * eps_truck / TL_truck

ev_ep_car = usdperkwh_blend_ep * eps_car / LF_car
ev_ep_bus = usdperkwh_blend_ep * eps_bus / LF_bus
ev_ep_trk = usdperkwh_blend_ep * eps_truck / TL_truck

print("EV infra CP (USD/pass-km): car, bus =", ev_cp_car, ev_cp_bus, "truck USD/ton-km =", ev_cp_trk)
print("EV infra EP (USD/pass-km): car, bus =", ev_ep_car, ev_ep_bus, "truck USD/ton-km =", ev_ep_trk)

EV infra CP (USD/pass-km): car, bus = 0.003180418571463413 0.0012139150634706097 truck USD/ton-km = 0.0040073274000439005
EV infra EP (USD/pass-km): car, bus = 0.0035610694358643676 0.0013592034297144312 truck USD/ton-km = 0.004486947489189103


In [29]:
LF_bus

12.837843

In [25]:
# --------------------------
# 6) H2 station subsidies (weighted avg; 백만원 -> constant USD base)
# --------------------------
df_h2 = build_h2_station_df_2022_2024()

yearly = yearly_weighted_average(
    df_h2,
    year_col="year",
    value_col="subsidy_baek",
    weight_col="n_sites",
).rename(columns={"total_weight": "total_sites", "wavg_value": "wavg_baek"})

usd_rows = []
for _, row in yearly.iterrows():
    y = int(row["year"])
    usd_base = baekmanwon_to_usd_base(
        float(row["wavg_baek"]),
        year=y,
        base_year=BASE_INPUT,
        dfExcAnnual=dfExcAnnual,
        arrDef=arrDef,
    )
    usd_rows.append((y, usd_base))

df_h2_usd = pd.DataFrame(usd_rows, columns=["year", "usd_per_station_base"]).sort_values("year")

cp_h2_usd_per_station = float(df_h2_usd["usd_per_station_base"].mean())
ep_idx = df_h2_usd["usd_per_station_base"].idxmax()
ep_h2_year = int(df_h2_usd.loc[ep_idx, "year"])
ep_h2_usd_per_station = float(df_h2_usd.loc[ep_idx, "usd_per_station_base"])

print(df_h2_usd)
print(f"H2 station CP (USD/station const {BASE_INPUT}):", cp_h2_usd_per_station)
print(f"H2 station EP (USD/station const {BASE_INPUT}), year {ep_h2_year}:", ep_h2_usd_per_station)

   year  usd_per_station_base
0  2022          1.231910e+06
1  2023          1.246598e+06
2  2024          1.452095e+06
H2 station CP (USD/station const 1990): 1310201.1414274608
H2 station EP (USD/station const 1990), year 2024: 1452095.4095439368


In [26]:
# --------------------------
# 7) H2 infra -> USD per service
# --------------------------
H2_ton_per_year_total = 15163
n_station_total = 437
n_station_subsidized = 56
L_h2 = 15

H2_kg_pv_total_network = (H2_ton_per_year_total * 1000.0) * pv_factor(L_h2, r)

cp_h2_total_usd_base = cp_h2_usd_per_station * n_station_subsidized
ep_h2_total_usd_base = ep_h2_usd_per_station * n_station_subsidized

cp_usd_per_kg = cp_h2_total_usd_base / H2_kg_pv_total_network
ep_usd_per_kg = ep_h2_total_usd_base / H2_kg_pv_total_network

KM_PER_KG_CAR, KM_PER_KG_BUS, KM_PER_KG_TRUCK = 95.73, 26.29, 16.61
kg_per_km_car = 1.0 / KM_PER_KG_CAR
kg_per_km_bus = 1.0 / KM_PER_KG_BUS
kg_per_km_trk = 1.0 / KM_PER_KG_TRUCK

h2_cp_car = cp_usd_per_kg * kg_per_km_car / LF_car
h2_cp_bus = cp_usd_per_kg * kg_per_km_bus / LF_bus
h2_cp_trk = cp_usd_per_kg * kg_per_km_trk / TL_truck

h2_ep_car = ep_usd_per_kg * kg_per_km_car / LF_car
h2_ep_bus = ep_usd_per_kg * kg_per_km_bus / LF_bus
h2_ep_trk = ep_usd_per_kg * kg_per_km_trk / TL_truck

print("H2 infra CP: car/bus USD/pass-km =", h2_cp_car, h2_cp_bus, "truck USD/ton-km =", h2_cp_trk)
print("H2 infra EP: car/bus USD/pass-km =", h2_ep_car, h2_ep_bus, "truck USD/ton-km =", h2_ep_trk)


H2 infra CP: car/bus USD/pass-km = 0.003735393424796729 0.0013349724638892987 truck USD/ton-km = 0.006458564946823436
H2 infra EP: car/bus USD/pass-km = 0.004139935063007456 0.0014795494564820873 truck USD/ton-km = 0.007158024989434746


In [27]:
# --------------------------
# 8) Build items -> merge -> XML
# --------------------------
SUBSIDY_NAME = "charging-infra-subsidy"

df_ev_infra_bev_cp  = pd.DataFrame({"subsidy_per_perf": [ev_cp_car, ev_cp_bus, ev_cp_trk]}, index=["Car", "Bus", "Medium truck"])
df_ev_infra_fcev_cp = pd.DataFrame({"subsidy_per_perf": [0.0, 0.0, 0.0]},             index=["Car", "Bus", "Medium truck"])

df_ev_infra_bev_ep  = pd.DataFrame({"subsidy_per_perf": [ev_ep_car, ev_ep_bus, ev_ep_trk]}, index=["Car", "Bus", "Medium truck"])
df_ev_infra_fcev_ep = pd.DataFrame({"subsidy_per_perf": [0.0, 0.0, 0.0]},             index=["Car", "Bus", "Medium truck"])

df_h2_infra_bev_cp  = pd.DataFrame({"subsidy_per_perf": [0.0, 0.0, 0.0]},             index=["Car", "Bus", "Medium truck"])
df_h2_infra_fcev_cp = pd.DataFrame({"subsidy_per_perf": [h2_cp_car, h2_cp_bus, h2_cp_trk]}, index=["Car", "Bus", "Medium truck"])

df_h2_infra_bev_ep  = pd.DataFrame({"subsidy_per_perf": [0.0, 0.0, 0.0]},             index=["Car", "Bus", "Medium truck"])
df_h2_infra_fcev_ep = pd.DataFrame({"subsidy_per_perf": [h2_ep_car, h2_ep_bus, h2_ep_trk]}, index=["Car", "Bus", "Medium truck"])

items_cp = merge_items_sum(
    items_from_perf_df(df_ev_infra_bev_cp, df_ev_infra_fcev_cp, subsidy_name=SUBSIDY_NAME)
    + items_from_perf_df(df_h2_infra_bev_cp, df_h2_infra_fcev_cp, subsidy_name=SUBSIDY_NAME)
)

items_ep = merge_items_sum(
    items_from_perf_df(df_ev_infra_bev_ep, df_ev_infra_fcev_ep, subsidy_name=SUBSIDY_NAME)
    + items_from_perf_df(df_h2_infra_bev_ep, df_h2_infra_fcev_ep, subsidy_name=SUBSIDY_NAME)
)

records_cp = make_records(items_cp, years=YEARS_XML, negative=True)
records_ep = make_records(items_ep, years=YEARS_XML, negative=True)

cp_xml_path = "../../input/policy/korea-2035/transportation/ZEV_charging_infra_subsidy_cp.xml"
ep_xml_path = "../../input/policy/korea-2035/transportation/ZEV_charging_infra_subsidy_ep.xml"

records_to_gcam_xml(records_cp, cp_xml_path)
records_to_gcam_xml(records_ep, ep_xml_path)

print("Wrote:")
print(" -", cp_xml_path)
print(" -", ep_xml_path)

Wrote:
 - ../../input/policy/korea-2035/transportation/ZEV_charging_infra_subsidy_cp.xml
 - ../../input/policy/korea-2035/transportation/ZEV_charging_infra_subsidy_ep.xml
